## 6) Tool Calling

LLMs are great at writing poetry, but they are often bad at math and they don't know the current weather.
Tool calling is like giving the AI **specialized gadgets** to help it do its job.
You tell the AI: *"I am giving you a calculator tool. If you see a math problem, don't guess—use the tool."*
The AI decides **when** it needs help and **which** gadget to pick to solve the problem.

###  Code Context
In the code below:
* `@tool`: This is a decorator. It puts a "sticker" on your Python function that says *"Hey AI, you are allowed to use this!"*
* `llm.bind_tools(tools)`: This effectively hands the manual of available tools to the "Brain."
* **Important:** When the model uses a tool, it doesn't return a text answer immediately. It returns a `tool_call` (a request to run the code). You (the code) must then run that function and give the answer back to the AI.

📚 **[Docs: Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)**


![Diagram](./images/06_img.png)

In [1]:
import os
import sys
from pathlib import Path

print(sys.executable)
project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week1-langchain/.venv/bin/python
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.5     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.6     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packag

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", temperature=0)


In [4]:
from langchain.tools import tool
from langchain.messages import HumanMessage
import math

@tool
def circle_area(radius: float) -> float:
    """Calculate the area of a circle given its radius."""
    return math.pi * radius ** 2

tools = [circle_area]

## Bind tools to the model so it can decide to call them
llm_with_tools = llm.bind_tools(tools)

msg = HumanMessage(content="I have a circular garden of radius 3. What's the area?")
ai_response = llm_with_tools.invoke([msg])

print("Model output Type", type(ai_response))
print("Model output", ai_response)
print("Tool calls (if any):", getattr(ai_response, "tool_calls", None))
print(ai_response.content)

Model output Type <class 'langchain_core.messages.ai.AIMessage'>
Model output content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 61, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f19e0a6e53', 'id': 'chatcmpl-EYcwJ7Y0X4dvy7tcRGzBQLeF6dCZB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a136d8-d4e9-77b3-b1c9-571bfcc4e71b-0' tool_calls=[{'name': 'circle_area', 'args': {'radius': 3}, 'id': 'call_VbEugXDBqnnyjbViEuL1V2un', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 61, 'output_tokens'

In [5]:
final = None
if getattr(ai_response, "tool_calls", None):
    for tool_call in ai_response.tool_calls:
        if tool_call["name"] == "circle_area":
            r = float(tool_call["args"]["radius"])
            result = circle_area.invoke({"radius": r})
            final = f"The area is approximately {result:.2f} square units."
else:
    final = ai_response.content

print("Final:", final)

Final: The area is approximately 28.27 square units.
